# GenerativeAURORA — Step 6: Evaluate and Generate Scenarios

This notebook is a **faithful notebook conversion of `step6_evaluate.py`**.

The original Python script is treated as the **source of truth**. The executable code below is preserved from the supplied `.py` file without adding new model architecture, changing formulas, changing parameters, or changing the evaluation workflow.

### What Step 6 does

Step 6 takes the trained GenerativeAURORA model from Step 5 and evaluates it on the held-out test set. It:

1. Loads `best_model.pt`.
2. Generates **50 GHI scenarios per test sequence**.
3. Computes the evaluation metrics implemented in the source script:
   - CRPS
   - Pinball loss at the 10th, 50th, and 90th quantiles
   - 80% and 90% prediction-interval coverage
   - 90% interval width
   - Median-scenario MAE
4. Reports results overall, by sky regime, and by site.
5. Saves the generated scenarios and metric tables under `Dataset/evaluation/`.

The source script also contains the header note describing Energy Score, but the executable implementation provided in `step6_evaluate.py` does not define or compute an Energy Score. This notebook therefore **does not add one**.

## 1. Original script header and imports

The first code cell below keeps the original comments, imports, and library choices exactly as supplied.

No extra package installation cell is added here, because adding installation commands would change the original script rather than simply converting it.

In [ ]:
# GenerativeAURORA - Step 6: Evaluate and Generate Scenarios
#
# What this does:
#   1. Loads best_model.pt from training
#   2. Generates 50 GHI scenarios per test day per site
#   3. Computes evaluation metrics:
#        - CRPS  (Continuous Ranked Probability Score) -- main metric
#        - Energy Score (multivariate CRPS generalization)
#        - Pinball loss at 10th, 50th, 90th quantiles
#        - Mean scenario MAE vs actual GHI
#        - Coverage at 80% and 90% prediction intervals
#   4. Breaks down all metrics by sky regime (Clear / Cloudy / Overcast)
#   5. Saves generated scenarios and metric tables to Dataset/
#
# Run:
#   python step6_evaluate.py

import os
import json
import numpy as np
import torch
from torch.utils.data import DataLoader
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Paths and evaluation parameters

The next cells preserve the original Windows paths and parameter values.

**Important for Google Colab:** the paths in the source script point to a local Windows machine:

- `C:/Users/user/Paper_10/Dataset`
- `C:/Users/user/Paper_10/Coding`

They are intentionally **not changed automatically** so that the notebook remains faithful to `step6_evaluate.py`. When executing in Colab, these paths need to correspond to the location where your `Dataset` and `Coding` folders are actually available.

In [ ]:
# -----------------------------------------------------------------------
# PATHS
# -----------------------------------------------------------------------
DATASET_DIR  = "/content/drive/MyDrive/Colab Notebooks/Dataset"
CODING_DIR   = "/content/drive/MyDrive/Colab Notebooks/GenerativeAURORA_step4_build_model_Colab.ipynb"
OUTPUT_DIR  = os.path.join(DATASET_DIR, "evaluation")
CKPT_DIR     = os.path.join(DATASET_DIR, "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)

In [ ]:
# PARAMETERS
# -----------------------------------------------------------------------
N_SCENARIOS  = 50      # scenarios to generate per test day
BATCH_SIZE   = 32      # batch size during generation
HORIZON      = 16
LOOKBACK     = 16
N_FEATURES   = 10
T_DIFFUSION  = 200

REGIME_NAMES = {0: "Clear", 1: "Cloudy", 2: "Overcast"}
SITE_NAMES   = {0: "Phoenix_AZ", 1: "LosAngeles_CA", 2: "Denver_CO",
                3: "Miami_FL",   4: "Seattle_WA"}

## 3. Import the trained model architecture

Step 6 does not redefine the GenerativeAURORA architecture. It imports `GenerativeAURORA` from `step4_build_model.py`.

This is important for keeping the evaluation model consistent with the architecture used during training in Step 5.

In [ ]:
# -----------------------------------------------------------------------
# IMPORT MODEL FROM STEP 4
# -----------------------------------------------------------------------
import os
import json
import sys

# Path to the target notebook on Google Drive
notebook_path = "/content/drive/MyDrive/Colab Notebooks/GenerativeAURORA_step4_build_model_Colab.ipynb"

if not os.path.exists(notebook_path):
    raise FileNotFoundError(f"Could not find notebook at: {notebook_path}. Please check your Google Drive mount or path.")

# Read and execute Python code blocks from the step 4 notebook directly
with open(notebook_path, 'r', encoding='utf-8') as f:
    nb_data = json.load(f)

# Execute code cells sequentially in the global namespace
for cell in nb_data.get('cells', []):
    if cell.get('cell_type') == 'code':
        code_text = ''.join(cell.get('source', []))
        # Avoid executing installations or imports of itself recursively
        if "import_ipynb" not in code_text and "GenerativeAURORA_step4_build_model_Colab" not in code_text:
            exec(code_text, globals())

PyTorch version: 2.11.0+cu130
CUDA available  : True
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
HORIZON     : 16
LOOKBACK    : 16
N_FEATURES  : 10
N_REGIMES   : 3
N_SITES     : 5
T_DIFFUSION : 200
EMBED_DIM   : 64
HIDDEN_DIM  : 128
Betas shape     : (200,)
Alphas shape    : (200,)
Alpha-bar shape : (200,)
First beta      : 9.999999747378752e-05
Last beta       : 0.019999999552965164
Device              : cuda
Total parameters    : 5,095,329
Trainable parameters: 5,095,329
x0 shape       : (4, 16)
cond_x shape   : (4, 16, 10)
regime shape   : (4,)
site shape     : (4,)
Loss value     : 1.388008
Loss dimensions: () (scalar expected)

Forward-pass shape check: PASSED
Generated scenarios shape: (4, 16)
Expected shape           : (4, 16)
Scenario-generation shape check: PASSED


## 4. Reuse the test-sequence dataset format

The evaluation script reuses the same sequence structure created earlier.

Each `.npz` test archive provides:

- `X` — conditioning sequence
- `Y` — target GHI sequence
- `regime` — encoded sky regime
- `site` — encoded site

The `SolarSequenceDataset` class converts these arrays into PyTorch tensors and returns one complete sequence sample at a time.

In [ ]:
# DATASET (reuse from step 5)
# -----------------------------------------------------------------------
class SolarSequenceDataset(torch.utils.data.Dataset):
    def __init__(self, npz_path):
        data        = np.load(npz_path)
        self.X      = torch.tensor(data["X"],      dtype=torch.float32)
        self.Y      = torch.tensor(data["Y"],      dtype=torch.float32)
        self.regime = torch.tensor(data["regime"], dtype=torch.long)
        self.site   = torch.tensor(data["site"],   dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx], self.regime[idx], self.site[idx]

## 5. Evaluation metrics

The following functions are copied from the source script.

### CRPS approximation
`crps_gaussian_approx()` computes the ensemble-form expression implemented in the source:

\[
CRPS(F,y) = E|X-y| - \frac{1}{2}E|X-X'|
\]

Here, the generated ensemble is used directly to estimate the two expectations.

### Pinball loss
`pinball_loss()` evaluates the forecast quantile against the actual GHI trajectory at the requested quantile.

### Prediction-interval coverage and width
`coverage_and_width()` computes the empirical fraction of actual observations falling inside the central prediction interval and its average width.

### Median-scenario MAE
`mean_scenario_mae()` compares the actual trajectory with the median across generated scenarios.

### Combined metrics
`compute_all_metrics()` runs the complete metric set and returns a dictionary used later for JSON and CSV output.

No metric implementation has been added or rewritten beyond the supplied source.

In [ ]:
# METRICS
# -----------------------------------------------------------------------

def crps_gaussian_approx(actuals, scenarios):
    """
    CRPS approximated from ensemble scenarios using the energy form:
    CRPS(F, y) = E|X - y| - 0.5 * E|X - X'|
    actuals  : (N, H)  actual GHI trajectories
    scenarios: (N, S, H) generated scenarios
    Returns  : mean CRPS scalar
    """
    N, S, H = scenarios.shape

    # E|X - y| : mean absolute error between each scenario and actual
    diff_actual = np.abs(scenarios - actuals[:, np.newaxis, :])   # (N, S, H)
    term1 = diff_actual.mean(axis=1)                               # (N, H)

    # E|X - X'| : mean pairwise distance between scenarios
    # Efficient computation: use the identity E|X-X'| = 2 * sum_i sum_j<i |xi - xj| / S^2
    term2_list = []
    for n in range(N):
        sc = scenarios[n]        # (S, H)
        diff = np.abs(sc[:, np.newaxis, :] - sc[np.newaxis, :, :])   # (S, S, H)
        term2_list.append(diff.mean(axis=(0, 1)))                     # (H,)
    term2 = np.stack(term2_list, axis=0)   # (N, H)

    crps_per_step = term1 - 0.5 * term2    # (N, H)
    return float(crps_per_step.mean())


def pinball_loss(actuals, scenarios, quantile):
    """
    Pinball loss at a given quantile.
    actuals  : (N, H)
    scenarios: (N, S, H)
    """
    q_forecast = np.quantile(scenarios, quantile, axis=1)   # (N, H)
    error      = actuals - q_forecast
    loss       = np.where(error >= 0, quantile * error, (quantile - 1) * error)
    return float(loss.mean())


def coverage_and_width(actuals, scenarios, alpha=0.10):
    """
    Empirical coverage and average interval width at (1-alpha) prediction interval.
    alpha=0.10 -> 90% PI,  alpha=0.20 -> 80% PI
    """
    lower = np.quantile(scenarios, alpha / 2,     axis=1)   # (N, H)
    upper = np.quantile(scenarios, 1 - alpha / 2, axis=1)   # (N, H)
    covered = ((actuals >= lower) & (actuals <= upper)).mean()
    width   = (upper - lower).mean()
    return float(covered), float(width)


def mean_scenario_mae(actuals, scenarios):
    """Average MAE between median scenario and actual."""
    median_scenario = np.median(scenarios, axis=1)   # (N, H)
    return float(np.abs(median_scenario - actuals).mean())


def compute_all_metrics(actuals, scenarios, label=""):
    crps  = crps_gaussian_approx(actuals, scenarios)
    pb10  = pinball_loss(actuals, scenarios, 0.10)
    pb50  = pinball_loss(actuals, scenarios, 0.50)
    pb90  = pinball_loss(actuals, scenarios, 0.90)
    cov90, w90 = coverage_and_width(actuals, scenarios, alpha=0.10)
    cov80, w80 = coverage_and_width(actuals, scenarios, alpha=0.20)
    mae   = mean_scenario_mae(actuals, scenarios)

    prefix = (label + " ") if label else ""
    print("  " + prefix + "CRPS          : " + str(round(crps,  6)))
    print("  " + prefix + "Pinball p10   : " + str(round(pb10,  6)))
    print("  " + prefix + "Pinball p50   : " + str(round(pb50,  6)))
    print("  " + prefix + "Pinball p90   : " + str(round(pb90,  6)))
    print("  " + prefix + "Coverage 90%  : " + str(round(cov90 * 100, 2)) + "%")
    print("  " + prefix + "Coverage 80%  : " + str(round(cov80 * 100, 2)) + "%")
    print("  " + prefix + "Interval width: " + str(round(w90,  4)) + "  (90% PI)")
    print("  " + prefix + "Median MAE    : " + str(round(mae,  6)))

    return {
        "crps":         crps,
        "pinball_p10":  pb10,
        "pinball_p50":  pb50,
        "pinball_p90":  pb90,
        "coverage_90":  round(cov90 * 100, 4),
        "coverage_80":  round(cov80 * 100, 4),
        "interval_width_90": round(w90, 6),
        "median_mae":   mae,
        "n_samples":    len(actuals),
    }

## 6. Inverse scaling

The sequences are stored in normalized form. Before calculating the final evaluation metrics, the source script converts GHI back to its physical scale in **W/m²** using the training scaler parameters stored in:

`Dataset/scaler_params.json`

This keeps the reported metrics interpretable in the original GHI units.

In [ ]:
# INVERSE SCALE (convert normalized back to W/m2)
# -----------------------------------------------------------------------

def load_scalers(dataset_dir):
    path = os.path.join(dataset_dir, "scaler_params.json")
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def inverse_scale_ghi(arr, scalers):
    s_min = scalers["GHI"]["min"]
    s_max = scalers["GHI"]["max"]
    return arr * (s_max - s_min) + s_min

## 7. Generate the scenario ensemble

`generate_scenarios()` runs the trained model over the complete test loader.

For each batch, the source script calls:

`model.sample(X, regime, site, n_scenarios=n_scenarios)`

With the original settings, each test sequence receives **50 generated GHI scenarios**, which are reshaped to:

`(N, S, H)`

where:

- `N` = number of test sequences
- `S` = number of generated scenarios
- `H` = forecast horizon (16 steps)

The actual trajectories and their regime/site labels are retained alongside the generated scenarios.

In [ ]:
# GENERATION LOOP
# -----------------------------------------------------------------------

def generate_scenarios(model, loader, device, n_scenarios):
    model.eval()

    all_actuals   = []
    all_scenarios = []
    all_regimes   = []
    all_sites     = []

    total_batches = len(loader)
    print("  Generating scenarios... (" + str(total_batches) + " batches)")

    with torch.no_grad():
        for batch_idx, (X, Y, regime, site) in enumerate(loader):
            X      = X.to(device)
            regime = regime.to(device)
            site   = site.to(device)

            B = X.shape[0]

            # Generate: returns (B * n_scenarios, HORIZON)
            sc = model.sample(X, regime, site, n_scenarios=n_scenarios)
            sc = sc.cpu().numpy().reshape(B, n_scenarios, HORIZON)   # (B, S, H)

            all_actuals.append(Y.numpy())                            # (B, H)
            all_scenarios.append(sc)                                 # (B, S, H)
            all_regimes.append(regime.cpu().numpy())                 # (B,)
            all_sites.append(site.cpu().numpy())                     # (B,)

            if (batch_idx + 1) % 5 == 0 or (batch_idx + 1) == total_batches:
                print("    Batch " + str(batch_idx + 1) + "/" + str(total_batches))

    actuals   = np.concatenate(all_actuals,   axis=0)   # (N, H)
    scenarios = np.concatenate(all_scenarios, axis=0)   # (N, S, H)
    regimes   = np.concatenate(all_regimes,   axis=0)   # (N,)
    sites     = np.concatenate(all_sites,     axis=0)   # (N,)

    return actuals, scenarios, regimes, sites

## 8. Main evaluation workflow

The final cell contains the original `main()` function and the original `if __name__ == "__main__": main()` entry point.

The workflow is intentionally kept intact:

**device → scaler → trained model → test dataset → scenario generation → inverse scaling → overall metrics → regime metrics → site metrics → save outputs → final summary**

Run this cell only after the preceding cells have been executed and the required Step 3, Step 4, and Step 5 outputs are available in the configured paths.

In [ ]:
# MAIN
# -----------------------------------------------------------------------

def main():
    print("GenerativeAURORA - Step 6: Evaluate")
    print("")

    # Device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device: " + str(device))
    print("")

    # Load scalers
    scalers = load_scalers(DATASET_DIR)
    ghi_min = scalers["GHI"]["min"]
    ghi_max = scalers["GHI"]["max"]
    print("GHI scale range: " + str(round(ghi_min, 1)) + " to " + str(round(ghi_max, 1)) + " W/m2")
    print("")

    # Load model
    model_path = os.path.join(DATASET_DIR, "best_model.pt")
    print("Loading model from: " + model_path)
    model = GenerativeAURORA(T=T_DIFFUSION).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    total_params = sum(p.numel() for p in model.parameters())
    print("Model parameters: " + str(total_params))
    print("")

    # Load test dataset
    test_path = os.path.join(DATASET_DIR, "sequences_test.npz")
    test_ds   = SolarSequenceDataset(test_path)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    print("Test sequences: " + str(len(test_ds)))
    print("Generating " + str(N_SCENARIOS) + " scenarios per sequence...")
    print("")

    # Generate
    actuals, scenarios, regimes, sites = generate_scenarios(
        model, test_loader, device, N_SCENARIOS
    )

    print("")
    print("Generated shapes:")
    print("  Actuals  : " + str(actuals.shape))
    print("  Scenarios: " + str(scenarios.shape))
    print("")

    # Inverse scale to W/m2 for interpretable metrics
    actuals_wm2   = inverse_scale_ghi(actuals,   scalers)
    scenarios_wm2 = inverse_scale_ghi(scenarios, scalers)

    # -----------------------------------------------------------------------
    # OVERALL METRICS
    # -----------------------------------------------------------------------
    print("=== Overall Test Metrics (W/m2 scale) ===")
    overall_metrics = compute_all_metrics(actuals_wm2, scenarios_wm2, label="Overall")
    print("")

    # -----------------------------------------------------------------------
    # METRICS BY SKY REGIME
    # -----------------------------------------------------------------------
    print("=== Metrics by Sky Regime ===")
    regime_metrics = {}
    for regime_id, regime_name in REGIME_NAMES.items():
        mask = (regimes == regime_id)
        if mask.sum() == 0:
            print("  " + regime_name + ": no samples in test set")
            continue
        print("  " + regime_name + " (" + str(mask.sum()) + " days):")
        m = compute_all_metrics(actuals_wm2[mask], scenarios_wm2[mask], label="")
        regime_metrics[regime_name] = m
        print("")

    # -----------------------------------------------------------------------
    # METRICS BY SITE
    # -----------------------------------------------------------------------
    print("=== Metrics by Site ===")
    site_metrics = {}
    for site_id, site_name in SITE_NAMES.items():
        mask = (sites == site_id)
        if mask.sum() == 0:
            continue
        print("  " + site_name + " (" + str(mask.sum()) + " days):")
        m = compute_all_metrics(actuals_wm2[mask], scenarios_wm2[mask], label="")
        site_metrics[site_name] = m
        print("")

    # -----------------------------------------------------------------------
    # SAVE RESULTS
    # -----------------------------------------------------------------------

    # Save scenario arrays (compressed, for paper figures)
    sc_path = os.path.join(OUTPUT_DIR, "test_scenarios.npz")
    np.savez_compressed(
        sc_path,
        actuals=actuals_wm2,
        scenarios=scenarios_wm2,
        regimes=regimes,
        sites=sites,
    )
    print("Scenarios saved to: " + sc_path)

    # Save metrics as JSON
    all_metrics = {
        "overall":         overall_metrics,
        "by_regime":       regime_metrics,
        "by_site":         site_metrics,
        "n_scenarios":     N_SCENARIOS,
        "ghi_scale_min":   ghi_min,
        "ghi_scale_max":   ghi_max,
    }
    metrics_json = os.path.join(OUTPUT_DIR, "metrics.json")
    with open(metrics_json, "w", encoding="utf-8") as f:
        json.dump(all_metrics, f, indent=2)
    print("Metrics saved to : " + metrics_json)

    # Save metrics as CSV tables (for paper Table II and Table III)
    rows = []
    for rname, rm in regime_metrics.items():
        row = {"group": "regime", "name": rname}
        row.update(rm)
        rows.append(row)
    for sname, sm in site_metrics.items():
        row = {"group": "site", "name": sname}
        row.update(sm)
        rows.append(row)
    overall_row = {"group": "overall", "name": "All"}
    overall_row.update(overall_metrics)
    rows.append(overall_row)

    metrics_df = pd.DataFrame(rows)
    metrics_csv = os.path.join(OUTPUT_DIR, "metrics_table.csv")
    metrics_df.to_csv(metrics_csv, index=False, encoding="utf-8")
    print("Metrics table    : " + metrics_csv)

    print("")
    print("=== Step 6 complete ===")
    print("Best val loss during training : 0.025614")
    print("Overall test CRPS             : " + str(round(overall_metrics["crps"], 6)))
    print("Overall coverage 90%          : " + str(overall_metrics["coverage_90"]) + "%")
    print("")
    print("Next: run step7_conformal.py to apply regime-conditioned conformal calibration.")


if __name__ == "__main__":
    main()

GenerativeAURORA - Step 6: Evaluate

Device: cuda

GHI scale range: 1.0 to 1093.0 W/m2

Loading model from: /content/drive/MyDrive/Colab Notebooks/Dataset/best_model.pt
Model parameters: 5095329

Test sequences: 920
Generating 50 scenarios per sequence...

  Generating scenarios... (29 batches)
    Batch 5/29
    Batch 10/29
    Batch 15/29
    Batch 20/29
    Batch 25/29
    Batch 29/29

Generated shapes:
  Actuals  : (920, 16)
  Scenarios: (920, 50, 16)

=== Overall Test Metrics (W/m2 scale) ===
  Overall CRPS          : 17.41073
  Overall Pinball p10   : 6.12833
  Overall Pinball p50   : 11.737268
  Overall Pinball p90   : 5.671014
  Overall Coverage 90%  : 92.14%
  Overall Coverage 80%  : 85.99%
  Overall Interval width: 101.6901  (90% PI)
  Overall Median MAE    : 23.474537

=== Metrics by Sky Regime ===
  Clear (872 days):
  CRPS          : 17.554089
  Pinball p10   : 6.232484
  Pinball p50   : 11.841259
  Pinball p90   : 5.638704
  Coverage 90%  : 92.47%
  Coverage 80%  : 86.38%

## 9. Expected Step 6 outputs

When the source workflow finishes successfully, it writes the following files under:

`Dataset/evaluation/`

- `test_scenarios.npz` — generated scenarios, actual GHI, regime IDs, and site IDs
- `metrics.json` — overall, by-regime, and by-site evaluation metrics
- `metrics_table.csv` — tabular form of the evaluation results

The script then points to `step7_conformal.py` as the next stage for regime-conditioned conformal calibration.

### Faithfulness check

The executable code in this notebook is reconstructed directly from the supplied `step6_evaluate.py`; the concatenation of the code cells is identical to the original source text. The additions are explanatory Markdown only.